# Introduction
packages installation

In [1]:
! pip install pypsa highspy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 358.3/358.3 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.3/153.3 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.0/117.0 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 63.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 52.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.7/44.7 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 27.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 33.0 MB/s eta 0:00:00
ERROR: Operation cancelled by user


In [2]:
from google.colab import drive
drive.mount('/content/drive')


MessageError: Error: credential propagation was unsuccessful

In [ ]:
# Import packages
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pypsa
import highspy

ROOT_DIR = os.getcwd()
PROJECT_DIR = os.path.join(ROOT_DIR, "drive/MyDrive/Colab_Notebooks/ENNOH/Zonal_model/data")


# Input Data for Zone Modelling

In [ ]:
country_name = 'Slovenia'
year = 2040
country_code = 'SI00'   # Target country sheet
weather_profile = '2010.0'  # Historical weather shape to use


### Quantification - Annual Data


In [ ]:
# Load the sheet specifically for specific values
try:
    survey_file_path = os.path.join(PROJECT_DIR, "20230711-National_Trends_and_Energy_Mix_Survey.xlsx")

    df_country = pd.read_excel(survey_file_path, sheet_name=country_name)

    if not df_country.empty:
        print("✅",country_name,"annual demand data successfully imported from survey file.")

except Exception as e:
    print(f"❌ Error reading country sheet: {e}")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
elec_twh = 16.50
h2_twh = 0.581
methane_twh =  7.75

In [ ]:
annual_inputs = {
    'elec_annual_twh': elec_twh,
    'h2_annual_twh': h2_twh,
    'methane_demand_twh': methane_twh,
    'p2g_capacity_mw': 50,
    'biomethane_potential_twh': 0.45,
    'synthetic_methane_fraction': 0.05,
    'loss_factor': 0.06
}

print("--- Extracted Quantification Inputs ---")
for k, v in annual_inputs.items():
    print(f"{k}: {v}")

In [ ]:
# --- 2. STEP 1: DEMAND QUANTIFICATION ---
def quantify_total_demand(inputs):
    # Electricity: Adding Transmission & Distribution Losses
    total_elec = inputs['elec_annual_twh'] * (1 + inputs['loss_factor'])

    # Methane: Calculate the net gap after biomethane
    total_methane = inputs['methane_demand_twh']
    bio_methane = inputs['biomethane_potential_twh']
    net_methane_gap = total_methane - bio_methane

    # Hydrogen: Industrial base + Syn-Methane requirement (1 GWh CH4 requires ~1.05 GWh H2)
    # Note: Synthetic methane target is usually a % of the TOTAL methane demand
    h2_for_syn_methane = (total_methane * inputs['synthetic_methane_fraction']) * 1.05
    total_h2 = inputs['h2_annual_twh'] + h2_for_syn_methane

    return total_elec, total_h2, total_methane, net_methane_gap

total_elec, total_h2, total_methane, methane_gap = quantify_total_demand(annual_inputs)


In [ ]:
print("--- Step 1: Quantification Results ---")
print(f"Total Electricity (with losses): {total_elec:.2f} TWh")
print(f"Total Hydrogen Demand:          {total_h2:.3f} TWh")
print(f"Total Methane Demand:           {total_methane:.2f} TWh")
print(f"(=) Net Methane Gap:            {methane_gap:.2f} TWh")

### Generation Capacities

In [ ]:
# Capacities in Selected contry
try:
    Nat_trends_file_path = os.path.join(PROJECT_DIR, "PEMMDB_SI00_NationalTrends_2040.xlsx")
    xls_pemmdb = pd.ExcelFile(Nat_trends_file_path)

    # 1. Solar Extraction
    df_solar_raw = pd.read_excel(xls_pemmdb, sheet_name='Solar', header=None)
    solar_rooftop = float(df_solar_raw.iloc[9, 1]) * 1000
    solar_pv = float(df_solar_raw.iloc[8, 1]) * 1000
    solar_p_nom = solar_pv + solar_rooftop

    # 2. Wind Extraction
    df_wind_raw = pd.read_excel(xls_pemmdb, sheet_name='Wind', header=None)
    wind_p_nom = float(df_wind_raw.iloc[7, 1]) * 1000

    # 3. Hydro Extraction (RoR and Pondage)
    df_hydro = pd.read_excel(xls_pemmdb, sheet_name='Hydro', header=None)

    # Extract Run of River
    ror_row = df_hydro[df_hydro.iloc[:, 0].astype(str).str.contains('Run of River - Total turbining', case=False, na=False)]
    ror_p_nom = float(ror_row.iloc[0, 1]) if not ror_row.empty else 0.0

    # Extract Pondage
    pondage_row = df_hydro[df_hydro.iloc[:, 0].astype(str).str.contains('Pondage - Total turbining capacity', case=False, na=False)]
    pondage_p_nom = float(pondage_row.iloc[0, 1]) if not pondage_row.empty else 1088.3

    hydro_total_p_nom = ror_p_nom + pondage_p_nom

except Exception as e:
    print(f"❌ Error extracting from specific cells: {e}")

In [ ]:
# --- Battery Parameters
df_battery = pd.read_excel(xls_pemmdb, sheet_name='Battery', header=7)

df_battery.columns = df_battery.columns.str.strip()

# --- 2. RE-ESTABLISH BATTERY DATABASE ---
column_mapping = {
    'Net maximum capacity - generation perspective \n(MW)': 'Capacity (MW)',
    'Storage capacity \n(MWh)': 'Storage Capacity (MWh)',
    'Average efficiency': 'Average Efficiency',
    'Ramp up rate \n(MW/h)': 'Ramp Up Rate (MW/h)',
    'Ramp down rate \n(MW/h)': 'Ramp Down Rate (MW/h)'
}

# Selecting columns and dropping rows where Capacity is NaN
battery_database = df_battery[list(column_mapping.keys())].rename(columns=column_mapping)
battery_database = battery_database.dropna(subset=['Capacity (MW)'])
battery_database.insert(0, 'Country', 'SI')
battery_database.set_index('Country', inplace=True)

# Taking Slovenia (SI) value
p2g_cap = 50 # Default if not found, or extract from 'Electrolyser' sheet if exists



In [ ]:

# --- 1. LOAD DATA FROM EXCEL ---
try:
    # Define the input file path directly to ensure execution works


    # --- 3. OPTIMIZATION INPUTS CONSOLIDATION ---
    # Using total_elec and total_h2 from Step 1 (cell FCStHXRyIOic)
    optimization_inputs = {
        'country': country_name,
        'year': year,
        'annual_elec_demand_twh': total_elec,
        'annual_h2_demand_twh': total_h2,
        'p2g_target_mw': annual_inputs['p2g_capacity_mw'],
        'total_methane_demand_twh': annual_inputs['methane_demand_twh'],
        'biomethane_potential_twh': annual_inputs['biomethane_potential_twh'],
        'battery_capacity_mw': float(battery_database.loc['SI', 'Capacity (MW)']),
        'battery_storage_mwh': float(battery_database.loc['SI', 'Storage Capacity (MWh)']),
        'battery_efficiency': float(battery_database.loc['SI', 'Average Efficiency']),
        'synthetic_methane_share': annual_inputs['synthetic_methane_fraction']
    }

    print("--- Final Optimization Inputs Summary ---")
    for key, value in optimization_inputs.items():
        if isinstance(value, float):
            print(f"{key:25}: {value:.2f}")
        else:
            print(f"{key:25}: {value}")

except Exception as e:
    print(f"❌ Error preparing optimization inputs: {e}")

In [ ]:
# Extract pondage capacity from df_hydro

# --- Slovenia 2040 Generation Capacities (Original Version) ---
elec_generators = {
    "Solar":   {"p_nom": np.round(solar_p_nom,1), "marginal_cost": 0.01, "carrier": "solar"},
    "Nuclear": {"p_nom": 696.0,  "marginal_cost": 12.0, "carrier": "nuclear"},
    "Hydro":   {"p_nom": np.round(hydro_total_p_nom, 1), "marginal_cost": 0.5,  "carrier": "hydro"},
    "Coal":    {"p_nom": 540.0,  "marginal_cost": 95.0, "carrier": "coal"},
    "Wind":    {"p_nom": np.round(wind_p_nom,1),   "marginal_cost": 0.01, "carrier": "wind"}
}

print("✅ Generation capacities successfully restored:")
for name, info in elec_generators.items():
    print(f"- {name}: {info['p_nom']} MW")

### Solar and Wind Profile from PECD

In [ ]:
# Load profiles for Solar and Wind
try:
    solar_file_path = os.path.join(PROJECT_DIR, "PECD_LFSolarPV_2040_SI00_edition 2023.2.csv")
    wind_file_path = os.path.join(PROJECT_DIR, "PECD_Wind_Onshore_2040_SI00_edition 2023.2.csv")
    df_solar_values_8760 = pd.read_csv(solar_file_path, header=10, usecols=['2010.0']).iloc[:, 0].astype(float)
    df_wind_values_8760 = pd.read_csv(wind_file_path, header=10, usecols=['2010.0']).iloc[:, 0].astype(float)

    # 2. Prepare target DatetimeIndex for a leap year (2040)
    target_len = 8784 # Hours in a leap year
    time_index_leap = pd.date_range(start='2040-01-01 00:00', periods=target_len, freq='h')

    # 3. Function to adjust 8760-hour data to 8784-hour leap year data
    def adjust_profile_for_leap_year(profile_8760_values, target_time_index):
        # Convert to numpy array for easier manipulation
        profile_array = profile_8760_values.to_numpy()

        # Find the hour corresponding to 29th February in the target_time_index
        # In a leap year, Feb 29th is day 59 (0-indexed). So its hours are from index 59*24 to 60*24-1
        feb_29_start_hour_idx = 59 * 24 # 59th day (0-indexed)

        # Get the values for Feb 28th (the 58th day, 0-indexed) from the 8760-hour profile
        feb_28_start_hour_idx_8760 = 58 * 24
        feb_28_end_hour_idx_8760 = (58 * 24) + 24

        feb_28_data_for_duplication = profile_array[feb_28_start_hour_idx_8760 : feb_28_end_hour_idx_8760]

        # Create a new array for 8784 hours
        profile_8784_array = np.zeros(target_len)

        # Fill the new array:
        # Part 1: Before Feb 29th
        profile_8784_array[:feb_29_start_hour_idx] = profile_array[:feb_29_start_hour_idx]

        # Part 2: Insert duplicated Feb 28th data for Feb 29th
        profile_8784_array[feb_29_start_hour_idx : feb_29_start_hour_idx + 24] = feb_28_data_for_duplication

        # Part 3: After Feb 29th, shift remaining data by 24 hours
        profile_8784_array[feb_29_start_hour_idx + 24 :] = profile_array[feb_28_end_hour_idx_8760:]

        return pd.Series(profile_8784_array, index=target_time_index)

    df_solar = adjust_profile_for_leap_year(df_solar_values_8760, time_index_leap)
    df_wind = adjust_profile_for_leap_year(df_wind_values_8760, time_index_leap)

    print("✅ Solar and Wind profiles cleaned, adjusted for 2040 leap year, and NaNs removed.")
    print(f"df_solar length: {len(df_solar)} | Max: {df_solar.max():.4f} | NaNs: {df_solar.isna().sum()}")
    print(f"df_wind length: {len(df_wind)} | Max: {df_wind.max():.4f} | NaNs: {df_wind.isna().sum()}")

except KeyError:
    print("❌ Error: '2010.0' column not found. Please check available columns in the CSV.")
except Exception as e:
    print(f"❌ Error preparing profiles: {e}")

In [ ]:
# Solar and Wind graphs
if not df_solar.empty and not df_wind.empty:
    # Filter data for June
    solar_june = df_solar.loc['2040-06-01':'2040-06-30']
    wind_june = df_wind.loc['2040-06-01':'2040-06-30']

    plt.figure(figsize=(15, 6))
    plt.plot(solar_june.index, solar_june.values, label='Solar Capacity Factor', color='orange', alpha=0.7)
    plt.plot(wind_june.index, wind_june.values, label='Wind Onshore Capacity Factor', color='blue', alpha=0.7)

    plt.title('Slovenia 2040: Hourly Solar and Onshore Wind Profiles (June) - Note: Solar 2010.0 is Zero', fontsize=14)
    plt.xlabel('Date and Time', fontsize=12)
    plt.ylabel('Capacity Factor', fontsize=12)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    if df_solar.max() == 0:
        print("⚠️ Warning: The solar capacity factor for '2010.0' is entirely zero in the source file. This profile will not contribute to solar generation.")

else:
    print("Solar and wind data are not available to plot.")



## Hydro data

In [ ]:
try:
    hydro_file_path = os.path.join(PROJECT_DIR, "PEMMDB_SI00_Hydro_Inflows_2040.xlsx")
    xls_hydro = pd.ExcelFile(hydro_file_path)
    df_ror_uniform = pd.read_excel(xls_hydro, sheet_name='Run of River - Uniform', header=7)
    df_year_dep = pd.read_excel(xls_hydro, sheet_name='Run of River - Year Dependent', header=7)
    print("✅ Hydro Inflow sheets (Uniform and Year Dependent) loaded successfully.")
except Exception as e:
    print(f"❌ Error loading hydro sheets: {e}")

### Run-of-river

In [ ]:
# Extracting Global Max and Min from the identified valid data columns
try:
    # Using the columns identified in the diagnostic step (kVI_TkoSe3Mu)
    # 97.764... is the Max Turbine column, 2.773... is the Min Turbine column
    max_val = df_ror_uniform[97.7640232690727].astype(float).max()
    min_val = df_ror_uniform[2.773679375814548].astype(float).min()

    print(f"--- Turbine Constraints Summary (Uniform) ---")
    print(f"Maximum Turbine Capacity: {max_val:.2f} MW")
    print(f"Minimum Turbine Capacity: {min_val:.2f} MW")

    # Update global variables for use in generator definitions
    max_turb = max_val
    min_turb = min_val
except Exception as e:
    print(f"❌ Error calculating max/min: {e}")

In [ ]:
# 1. Define target length for Leap Year 2040 (366 days * 24 hours)

try:
    target_hours = 8784
    p_nom_hydro = 140.61566

    # 2. Repeat daily values 24 times
    # daily_max_mw and daily_min_mw were defined in Step 1 (cell 002ed608)
    max_hourly_mw = daily_max_mw.repeat(24).values
    min_hourly_mw = daily_min_mw.repeat(24).values

    # 3. Align to exactly 8784 hours (padding missing end-of-year days with 'edge' mode)
    max_p_pu = np.pad(max_hourly_mw, (0, max(0, target_hours - len(max_hourly_mw))), mode='edge')[:target_hours] / p_nom_hydro
    min_p_pu = np.pad(min_hourly_mw, (0, max(0, target_hours - len(min_hourly_mw))), mode='edge')[:target_hours] / p_nom_hydro

    # 4. Store final results
    p_max_pu_final_full = max_p_pu
    p_min_pu_final_full = min_p_pu

    print("✅ Hydro Turbine profiles (Min/Max) successfully processed and aligned to 8784 hours (366 days).")
    print(f"Final profile length: {len(p_max_pu_final_full)} hours.")

except Exception as e:
    print(f"❌ Error during hourly translation: {e}")

In [ ]:
# --- Hydro-Specific Profile Preparation ---
try:

    def force_align(data, target_len=8784):
        arr = np.array(data)
        return np.pad(arr, (0, max(0, target_len - len(arr))), mode='edge')[:target_len]

    # 2. Daily Turbine Constraints (from df_ror_uniform)
    daily_min_mw = df_ror_uniform.iloc[:, 7].astype(float)
    daily_max_mw = df_ror_uniform.iloc[:, 8].astype(float)

    # 3. 2010 Weather-Dependent Inflow
    # Ensure inflow data exists and handle potential NaNs by filling with 0
    if 'ror_inflow_2010_mw' in locals():
        inflow_cleaned = pd.Series(ror_inflow_2010_mw).fillna(0)
    else:
        # Fallback to zero inflow if the variable was not found
        inflow_cleaned = pd.Series(0, index=range(target_hours))
        print('⚠️ Warning: ror_inflow_2010_mw not found, defaulting to zero.')

    # 4. Converting daily extracted inflow to hourly per-unit
    p_max_pu_turb = force_align(daily_max_mw.repeat(24)) / p_nom_hydro
    p_min_pu_turb = force_align(daily_min_mw.repeat(24)) / p_nom_hydro
    p_inflow_pu = force_align(inflow_cleaned) / p_nom_hydro

    # 5. Final Effective Hydro Capability
    effective_p_max_pu = np.minimum(p_inflow_pu, p_max_pu_turb)

    # Define the exact variable names used in cell 62934984
    p_max_pu_final_full = effective_p_max_pu
    p_min_pu_final_full = p_min_pu_turb

    print(f'✅ Hydro profiles for 2040 successfully prepared.')
    print(f'Average Effective Capacity: {np.nanmean(p_max_pu_final_full):.4f} p.u.')

except Exception as e:
    print(f'❌ Error during hydro profile setup: {e}')

In [ ]:
# STEP 1: Identification and Extraction of 2010 Inflow Data
try:
    # Load the first 10 rows to find the '2010' header
    df_raw_years = pd.read_excel(xls_hydro, sheet_name='Run of River - Year Dependent', header=None, nrows=10)

    target_col_idx = None
    for i, row in df_raw_years.iterrows():
        matches = row[row == 2010].index.tolist()
        if matches:
            target_col_idx = matches[0]
            print(f"✅ Found '2010' on Row Index {i}, Column Index {target_col_idx}")
            break

    if target_col_idx is not None:
        # Extract daily GWh data starting from data row (index 5)
        daily_inflow_2010_gwh = pd.read_excel(xls_hydro, sheet_name='Run of River - Year Dependent', header=None).iloc[5:, target_col_idx].astype(float)

        # Convert GWh/day to average hourly MW
        daily_inflow_2010_mw = daily_inflow_2010_gwh * 1000 / 24

        # Translate to 8784-hour hourly profile (Leap Year 2040)
        target_hours = 8784
        inflow_hourly_raw = daily_inflow_2010_mw.repeat(24).values
        ror_inflow_2010_mw = np.pad(inflow_hourly_raw, (0, max(0, target_hours - len(inflow_hourly_raw))), mode='edge')[:target_hours]

        print(f"✅ Hourly profile for 2010 weather shape successfully created ({len(ror_inflow_2010_mw)} hours).")
    else:
        print("❌ Error: Could not find year 2010 in the spreadsheet headers.")

except Exception as e:
    print(f"❌ Error during 2010 inflow extraction: {e}")

In [ ]:
# 1. Align arrays to 8784 hours (Leap Year 2040)
time_index_leap = pd.date_range(start='2040-01-01', periods=8784, freq='h')

# 2. Extract Turbine Constraints from df_ror_uniform (Daily to Hourly)
# Column index 7 is MIN_TURB, Column index 8 is MAX_TURB
daily_min = df_ror_uniform.iloc[:, 7].astype(float)
daily_max = df_ror_uniform.iloc[:, 8].astype(float)

p_min_pu = np.pad(daily_min.repeat(24).values, (0, 8784), mode='edge')[:8784] / p_nom_hydro
p_max_pu = np.pad(daily_max.repeat(24).values, (0, 8784), mode='edge')[:8784] / p_nom_hydro

# 3. Convert River Inflow to per-unit
inflow_pu = ror_inflow_2010_mw / p_nom_hydro

# 4. Visualization (Sample: June)
plt.figure(figsize=(15, 7))
# For June (30 days * 24 hours = 720 hours)
# Start of June for 8784 hours (leap year): (31+29+31+30+31) * 24 = 152 * 24 = 3648 hours
# Correct index for June: 31 (Jan) + 29 (Feb) + 31 (Mar) + 30 (Apr) + 31 (May) = 152 days
# start_h = 152 * 24 = 3648
# end_h = start_h + (30 * 24) = 3648 + 720 = 4368
start_h, end_h = 3648, 4368

# Shading the dispatch possibility area
plt.fill_between(time_index_leap[start_h:end_h], p_min_pu[start_h:end_h], p_max_pu[start_h:end_h], color='lightblue', alpha=0.4, label='Dispatch Possibility Range')

plt.step(time_index_leap[start_h:end_h], p_max_pu[start_h:end_h], label='Max Turbine Limit (p.u.)', where='post', color='darkblue', linestyle='--', linewidth=1.5)
plt.step(time_index_leap[start_h:end_h], p_min_pu[start_h:end_h], label='Min Turbine Limit (p.u.)', where='post', color='skyblue',linestyle='--', linewidth=1.5)
plt.plot(time_index_leap[start_h:end_h], inflow_pu[start_h:end_h], label='River Inflow 2010 (p.u.)', color='aqua', alpha=0.8, linewidth=2.5)

plt.title('Slovenia 2040: Combined Hydro Run-of-River Constraints & Inflow (June)', fontsize=14)
plt.ylabel('Capacity Factor [p.u.]')
plt.legend(loc='upper left')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'Average 2010 Inflow: {np.nanmean(inflow_pu):.4f} p.u.')
print(f'Average Max Capacity: {np.nanmean(p_max_pu):.4f} p.u.')

### Pondage

In [ ]:
try:
    # 1. Load Pondage sheets
    df_pondage_uniform = pd.read_excel(xls_hydro, sheet_name='Pondage - Uniform', header=7)
    df_pondage_years = pd.read_excel(xls_hydro, sheet_name='Pondage - Year Dependent', header=None, nrows=10)
    print("✅ Pondage sheets loaded successfully.")

    # 2. Extract Turbine Constraints (col 7 is Min, 8 is Max based on standard format)
    daily_min_pondage = df_pondage_uniform.iloc[:, 7].astype(float)
    daily_max_pondage = df_pondage_uniform.iloc[:, 8].astype(float)

    print(f"--- Turbine Constraints Summary (Pondage) ---")
    print(f"Maximum Turbine Capacity: {daily_max_pondage.max():.2f} MW")
    print(f"Minimum Turbine Capacity: {daily_min_pondage.min():.2f} MW")

    # 3. Extract 2010 Inflow Data
    target_col_idx = None
    for i, row in df_pondage_years.iterrows():
        matches = row[row == 2010].index.tolist()
        if matches:
            target_col_idx = matches[0]
            break

    if target_col_idx is not None:
        df_pondage_inflow_full = pd.read_excel(xls_hydro, sheet_name='Pondage - Year Dependent', header=None)
        daily_inflow_gwh = df_pondage_inflow_full.iloc[5:, target_col_idx].astype(float)
        daily_inflow_mw = daily_inflow_gwh * 1000 / 24

        # 4. Convert to Hourly p.u. Arrays (8784 hours for Leap Year 2040)
        target_hours = 8784
        p_nom_pondage = 1088.3 # Nominal capacity from the database

        def force_align(data):
            arr = np.array(data)
            return np.pad(arr, (0, max(0, target_hours - len(arr))), mode='edge')[:target_hours]

        pondage_inflow_mw = force_align(daily_inflow_mw.repeat(24))

        p_max_pu_pondage = force_align(daily_max_pondage.repeat(24)) / p_nom_pondage
        p_min_pu_pondage = force_align(daily_min_pondage.repeat(24)) / p_nom_pondage
        p_inflow_pu_pondage = pondage_inflow_mw / p_nom_pondage

        print(f"✅ Hourly profiles for 2010 Pondage shape successfully created.")
        print(f"Average Inflow: {np.nanmean(p_inflow_pu_pondage):.4f} p.u.")
    else:
        print("❌ Error: Could not find year 2010 in Pondage sheet.")

except Exception as e:
    print(f"❌ Error during Pondage extraction: {e}")

In [ ]:
# Target time index for visualization (Leap year 2040)
time_index_leap = pd.date_range(start='2040-01-01', periods=8784, freq='h')

# Visualization (Sample: June)
plt.figure(figsize=(15, 7))
# Start of June for 8784 hours (leap year)
start_h, end_h = 3648, 4368

# Shading the dispatch possibility area
plt.fill_between(time_index_leap[start_h:end_h],
                 p_min_pu_pondage[start_h:end_h],
                 p_max_pu_pondage[start_h:end_h],
                 color='lightblue', alpha=0.4, label='Dispatch Possibility Range')

plt.step(time_index_leap[start_h:end_h], p_max_pu_pondage[start_h:end_h], label='Max Turbine Limit (p.u.)', where='post', color='darkblue', linestyle='--', linewidth=1.5)
plt.step(time_index_leap[start_h:end_h], p_min_pu_pondage[start_h:end_h], label='Min Turbine Limit (p.u.)', where='post', color='skyblue',linestyle='--', linewidth=1.5)
plt.plot(time_index_leap[start_h:end_h], p_inflow_pu_pondage[start_h:end_h], label='Pondage Inflow 2010 (p.u.)', color='aqua', alpha=0.8, linewidth=2.5)

plt.title('Slovenia 2040: Combined Hydro Pondage Constraints & Inflow (June)', fontsize=14)
plt.ylabel('Capacity Factor [p.u.]')
plt.legend(loc='upper left')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'Average 2010 Pondage Inflow: {np.nanmean(p_inflow_pu_pondage):.4f} p.u.')
print(f'Average Max Pondage Capacity: {np.nanmean(p_max_pu_pondage):.4f} p.u.')

### Combined Hydro Power Plants

In [ ]:

# --- Calculate Combined Capacity ---
p_nom_total_hydro = p_nom_hydro + p_nom_pondage

# --- Sum Megawatts ---
# Convert p.u. back to MW, sum them, and handle potential NaNs
total_max_mw = (p_max_pu_final_full * p_nom_hydro) + (p_max_pu_pondage * p_nom_pondage)
total_min_mw = (p_min_pu_final_full * p_nom_hydro) + (p_min_pu_pondage * p_nom_pondage)
# Using p_inflow_pu and pondage_inflow_mw (converted to p.u. earlier)
total_inflow_mw = (p_inflow_pu * p_nom_hydro) + (p_inflow_pu_pondage * p_nom_pondage)

# --- Convert back to combined per-unit (p.u.) profile for PyPSA ---
combined_p_max_pu = total_max_mw / p_nom_total_hydro
combined_p_min_pu = total_min_mw / p_nom_total_hydro
combined_p_inflow_pu = total_inflow_mw / p_nom_total_hydro

print(f"--- Aggregated Hydro Profile (RoR + Pondage) ---")
print(f"Total Combined Capacity: {p_nom_total_hydro:.2f} MW")
print(f"Average Combined Inflow: {np.nanmean(combined_p_inflow_pu):.4f} p.u.")

# --- Plot Combined Profile for June ---
plt.figure(figsize=(15, 6))
start_h, end_h = 3648, 4368  # June indices for 8784 hours

plt.fill_between(time_index_leap[start_h:end_h],
                 combined_p_min_pu[start_h:end_h],
                 combined_p_max_pu[start_h:end_h],
                 color='lightblue', alpha=0.3, label='Combined Dispatch Range (p.u.)')

plt.plot(time_index_leap[start_h:end_h], combined_p_inflow_pu[start_h:end_h],
         label='Combined Inflow (p.u.)', color='aqua', linewidth=3.0)

plt.step(time_index_leap[start_h:end_h], combined_p_max_pu[start_h:end_h],
         where='post', color='darkblue', linestyle='--', alpha=0.7, label='Max Limit')
plt.step(time_index_leap[start_h:end_h], combined_p_min_pu[start_h:end_h], label='Min Limit (p.u.)', where='post', color='skyblue',linestyle='--', linewidth=1.5)


plt.title('Slovenia 2040: Aggregated Hydro Constraints & Inflow (June)', fontsize=14)
plt.ylabel('Capacity Factor [p.u.]')
plt.legend(loc='upper left')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## Electricity Profile from Hystorical Profiles

In [ ]:
# Electricity Demand Profiles
try:
    # 1. Load the specific sheet
    demand_file_path = os.path.join(PROJECT_DIR, "2040_National Trends.xlsx")
    df_demand_full = pd.read_excel(demand_file_path, sheet_name=country_code)

    # 2. Identify the column for the selected weather year (located at row index 6)
    header_row = df_demand_full.iloc[6]
    try:
        # Exact float match
        col_idx = header_row[header_row == float(weather_profile)].index[0]
    except:
        # String-based fallback match
        col_idx = header_row[header_row.astype(str).str.contains(str(int(float(weather_profile))))].index[0]

    col_pos = df_demand_full.columns.get_loc(col_idx)

    # 3. Extract hourly data starting from row index 7
    raw_shape = df_demand_full.iloc[7:, col_pos].astype(float).reset_index(drop=True)

    # 4. Normalize and Scale to the 2040 annual electricity demand target (total_elec_si)
    # total_elec_si was calculated in cell FCStHXRyIOic
    target_mwh = total_elec * 1e6
    elec_profile_2040_scaled = (raw_shape / raw_shape.sum()) * target_mwh

    print(f"✅ Processed 2040 demand for {country_code} using {weather_profile} weather shape.")
    print(f"Annual Demand: {elec_profile_2040_scaled.sum()/1e6:.2f} TWh")
    print(f"Peak Demand:   {elec_profile_2040_scaled.max():.2f} MW")

except Exception as e:
    print(f"❌ Error processing demand for {country_code}: {e}")

In [ ]:
# Target length for Leap Year 2040
target_hours = 8784

# Convert to numpy array and pad to 8784 hours
# Using 'edge' mode to repeat the last values for the missing 24 hours
elec_values_8784 = np.pad(elec_profile_2040_scaled.values, (0, target_hours - len(elec_profile_2040_scaled)), mode='edge')

# Create a new Series with the correct leap year index
time_index_8784 = pd.date_range(start='2040-01-01 00:00', periods=target_hours, freq='h')
elec_profile_2040_scaled = pd.Series(elec_values_8784, index=time_index_8784)

print(f"\u2705 Electricity demand profile extended to {len(elec_profile_2040_scaled)} hours.")
print(f"New Annual Demand: {elec_profile_2040_scaled.sum()/1e6:.2f} TWh")

In [ ]:

# Filter for the month of June for visual verification
demand_june = elec_profile_2040_scaled.loc['2040-06-01':'2040-06-30']

plt.figure(figsize=(15, 6))
plt.plot(demand_june.index, demand_june.values, color='tab:brown', linewidth=1.2, label=f'Demand ({country_code})')

plt.title(f'Slovenia 2040: Hourly Electricity Demand (June, {weather_profile} Shape)', fontsize=14)
plt.ylabel('Demand [MW]', fontsize=12)
plt.xlabel('Date', fontsize=12)
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"June Statistics for {country_code}:")
print(f"- Max: {demand_june.max():.2f} MW")
print(f"- Avg: {demand_june.mean():.2f} MW")

# Presenting and Saving Data

In [ ]:
print("### Variables Used in Optimization ###\n")

print("#### 1. Dictionaries ####")
print("**a) `optimization_inputs` (Dictionary for General Parameters):**")
print("   Contains key parameters such as annual demands, p2g target, battery specs, and synthetic methane share.")
for k, v in optimization_inputs.items():
    if isinstance(v, (int, float)):
        print(f"   - {k}: {v:.2f}")
    else:
        print(f"   - {k}: {v}")

print("\n**b) `elec_generators` (Dictionary for Generator Capacities & Costs):**")
print("   Defines the nominal capacity (p_nom) and marginal cost for each electricity generator.")
for name, info in elec_generators.items():
    print(f"   - {name}: p_nom={info['p_nom']} MW, marginal_cost={info['marginal_cost']} EUR/MWh")


print("\n#### 2. Time Series Profiles (Pandas Series or NumPy Arrays) ####")

print("**a) `df_solar` (Solar Capacity Factor Profile):**")
print(f"   - Type: {type(df_solar)}")
print(f"   - Length: {len(df_solar)} hours (8784 hours, adjusted for leap year 2040)")
print("   - Description: Hourly capacity factor (0-1) for solar power generation.")

print("\n**b) `df_wind` (Wind Capacity Factor Profile):**")
print(f"   - Type: {type(df_wind)}")
print(f"   - Length: {len(df_wind)} hours (8784 hours, adjusted for leap year 2040)")
print("   - Description: Hourly capacity factor (0-1) for wind power generation.")

print("\n**c) `combined_p_max_pu`, `combined_p_min_pu`, `combined_p_inflow_pu` (Hydro Profiles):**")
print(f"   - Type: {type(combined_p_max_pu)}")
print(f"   - Length: {len(combined_p_max_pu)} hours (8784 hours, adjusted for leap year 2040)")
print("   - Description: Per-unit (p.u.) hourly profiles for combined Hydro (Run-of-River + Pondage)")
print("     - `combined_p_max_pu`: Maximum generation limit.")
print("     - `combined_p_min_pu`: Minimum generation limit.")
print("     - `combined_p_inflow_pu`: Inflow profile (effective availability).")

print("\n**d) `elec_profile_2040_scaled` (Electricity Demand Profile):**")
print(f"   - Type: {type(elec_profile_2040_scaled)}")
print(f"   - Length: {len(elec_profile_2040_scaled)} hours (8784 hours, adjusted for leap year 2040)")
print("   - Description: Hourly electricity demand in MW, scaled to the annual target.")

In [ ]:
import json
import os

# Combine the dictionaries into a single input_data dictionary
input_data = {
    'optimization_inputs': optimization_inputs,
    'elec_generators': elec_generators
}

# Define the file path to save the JSON file in the PROJECT_DIR
save_path = os.path.join(PROJECT_DIR, "input_data.json")

# Save the dictionary to a JSON file
try:
    with open(save_path, 'w') as f:
        json.dump(input_data, f, indent=4)
    print(f"✅ Successfully saved 'input_data.json' to {save_path}")
except Exception as e:
    print(f"❌ Error saving input_data to JSON: {e}")

In [ ]:
import pandas as pd
import os

# Ensure time_index_leap is available for NumPy arrays
if 'time_index_leap' not in locals():
    time_index_leap = pd.date_range(start='2040-01-01 00:00', periods=8784, freq='h')

# Create a dictionary of all profiles to combine
# Ensure all are pandas Series with the correct DatetimeIndex
profiles_to_combine = {
    'df_solar': df_solar,
    'df_wind': df_wind,
    'combined_p_max_pu': pd.Series(combined_p_max_pu, index=time_index_leap, name='combined_p_max_pu'),
    'combined_p_min_pu': pd.Series(combined_p_min_pu, index=time_index_leap, name='combined_p_min_pu'),
    'combined_p_inflow_pu': pd.Series(combined_p_inflow_pu, index=time_index_leap, name='combined_p_inflow_pu'),
    'elec_profile_2040_scaled': elec_profile_2040_scaled
}

# Combine all profiles into a single DataFrame
combined_profiles_df = pd.DataFrame(profiles_to_combine)

print("--- Combined Profiles DataFrame (First 3 rows) ---")
display(combined_profiles_df.head(3))

In [ ]:
# Save the combined DataFrame to a single CSV file with the requested name
combined_output_path = os.path.join(PROJECT_DIR, "input_profiles.csv") # Changed filename here
try:
    combined_profiles_df.to_csv(combined_output_path, index=True, header=True)
    print(f"\n✅ Successfully saved combined profiles to {combined_output_path}")
except Exception as e:
    print(f"❌ Error saving input_profiles.csv: {e}")


# Optimisation

## Network Definition and Busses

In [ ]:
# Slicing inputs for June only
june_snapshots = df_solar.loc['2040-06-01':'2040-06-30'].index


In [ ]:
import pypsa
import pandas as pd
import numpy as np

# --- 1. INITIALIZE NETWORK ---
n = pypsa.Network()

# Using simple integer indexing for 8784 hours instead of DateIndex
n.set_snapshots(june_snapshots)

# Define Carriers
for c in ["AC", "H2", "CH4", "CO2", "nuclear", "solar", "hydro", "coal", "wind"]:
    n.add("Carrier", c)

# --- 2. DEFINE BUSES ---
n.add("Bus", "SI_elec", carrier="AC")
n.add("Bus", "SI_h2", carrier="H2")
n.add("Bus", "SI_methane", carrier="CH4")
n.add("Bus", "SI_CO2", carrier="CO2")

### Electricity Generators

In [ ]:
# --- 3. ADD ELECTRICITY GENERATORS ---

# 3.1. Hydro (Combined RoR + Pondage as a single generator)
n.add("Generator", "SI_Hydro",
      bus="SI_elec",
      p_nom=elec_generators['Hydro']['p_nom'],
      p_max_pu=pd.Series(combined_p_max_pu, index=time_index_leap).loc[n.snapshots].values,
      p_min_pu=pd.Series(combined_p_min_pu, index=time_index_leap).loc[n.snapshots].values,
      marginal_cost=elec_generators['Hydro']['marginal_cost'],
      carrier="hydro",
      overwrite=True)

# 3.2. Nuclear (Must-run Baseload Constraint)
n.add("Generator", "SI_Nuclear",
      bus="SI_elec",
      p_nom=elec_generators['Nuclear']['p_nom'],
      marginal_cost=elec_generators['Nuclear']['marginal_cost'],
      p_min_pu=1.0,
      carrier="nuclear",
      overwrite=True)

# 3.3. Coal (with 24h Ramp Up/Down constraints)
# 1/24 means it takes 24 hours to ramp from 0 to p_nom
n.add("Generator", "SI_Coal",
      bus="SI_elec",
      p_nom=elec_generators['Coal']['p_nom'],
      marginal_cost=elec_generators['Coal']['marginal_cost'],
      ramp_limit_up=1/24,
      ramp_limit_down=1/24,
      p_min_pu=0.1,
      carrier="coal",
      overwrite=True)

# 3.4. Solar
n.add("Generator", "SI_Solar",
      bus="SI_elec",
      p_nom=elec_generators['Solar']['p_nom'],
      marginal_cost=elec_generators['Solar']['marginal_cost'],
      carrier="solar",
      p_max_pu=df_solar.loc[n.snapshots].values,
      overwrite=True)

# 3.5. Wind
n.add("Generator", "SI_Wind",
      bus="SI_elec",
      p_nom=elec_generators['Wind']['p_nom'],
      marginal_cost=elec_generators['Wind']['marginal_cost'],
      carrier="wind",
      p_max_pu=df_wind.loc[n.snapshots].values,
      overwrite=True)

### Storages

In [ ]:
# --- 4. ADD STORAGES ---
# battery
n.add("StorageUnit", "SI_battery",
      bus="SI_elec",
      p_nom=optimization_inputs['battery_capacity_mw'],
      max_hours=optimization_inputs['battery_storage_mwh'] / optimization_inputs['battery_capacity_mw'],
      efficiency_store=optimization_inputs['battery_efficiency'],
      efficiency_dispatch=optimization_inputs['battery_efficiency'],
      cyclic_state_of_charge=True,
      overwrite=True)

# H2 Storage
n.add("Store", "SI_h2_storage",
      bus="SI_h2", carrier="H2",
      e_nom_extendable=True, capital_cost=140,
      e_cyclic=True, overwrite=True)

# Methane storage
n.add("Store", "SI_methane_storage",
      bus="SI_methane", carrier="CH4",
      e_nom_extendable=True, capital_cost=15.0,
      e_cyclic=True, overwrite=True)

### Links

In [ ]:
# --- 5. SECTOR COUPLING LINKS ---
n.add("Link", "SI_gas_to_power", bus0="SI_methane", bus1="SI_elec", p_nom=520, efficiency=0.47, marginal_cost=2.0, overwrite=True)
n.add("Link", "SI_electrolyzer", bus0="SI_elec", bus1="SI_h2", p_nom=optimization_inputs['p2g_target_mw'], efficiency=0.76, p_nom_extendable=True, overwrite=True)
n.add("Link", "SI_SMR", bus0="SI_methane", bus1="SI_h2", p_nom=100, efficiency=0.72, marginal_cost=40, overwrite=True)
n.add("Link", "SI_SMR_with_CCS", bus0="SI_methane", bus1="SI_h2", p_nom_extendable=True, efficiency=0.68, marginal_cost=5, overwrite=True)
n.add("Link", "SI_h2_to_power", bus0="SI_h2", bus1="SI_elec", p_nom_extendable=True, efficiency=0.45, capital_cost=600000, overwrite=True)
n.add("Link", "SI_methanation", bus0="SI_h2", bus1="SI_methane", bus2="SI_CO2", efficiency=0.80, efficiency2=-0.2, capital_cost=500000, p_nom_extendable=True, overwrite=True)

# --- 6. BIOMETHANE ---
n.add("Bus", "SI_biomethane_source", carrier="CH4", overwrite=True)
n.add("Store", "SI_biomethane_reserve", bus="SI_biomethane_source", e_nom=optimization_inputs['biomethane_potential_twh'] * 1e6, e_initial=optimization_inputs['biomethane_potential_twh'] * 1e6, e_cyclic=False, overwrite=True)
n.add("Link", "SI_biomethane_injection", bus0="SI_biomethane_source", bus1="SI_methane", p_nom=1000, marginal_cost=55, overwrite=True)

# --- 7. ADD LOADS ---
# Adjust electricity load to match the network's snapshots (June)
n.add("Load", "SI_elec_load", bus="SI_elec", p_set=elec_profile_2040_scaled.loc[n.snapshots], overwrite=True)
# For H2 and Methane loads, if they are constant for the year, using the annual average for the shorter period is fine.
n.add("Load", "SI_h2_load", bus="SI_h2", p_set=optimization_inputs['annual_h2_demand_twh'] * 1e6 / 8784, overwrite=True)
n.add("Load", "SI_methane_load", bus="SI_methane", p_set=total_methane * 1e6 / 8784, overwrite=True)

# --- Add a dummy gas import to ensure feasibility ---
# This ensures methane demand can always be met, making the problem feasible
n.add("Generator", "SI_gas_imports",
      bus="SI_methane",
      p_nom=10000, # Large capacity to cover any deficit
      marginal_cost=100.0, # High marginal cost so it's only used if necessary
      carrier="CH4",
      overwrite=True)

### Optimise

In [ ]:
# --- 8. OPTIMIZE ---

# Only clip generators that have a time-varying p_min_pu defined
for gen in n.generators_t.p_min_pu.columns:
    if gen in n.generators_t.p_max_pu.columns:
        n.generators_t.p_max_pu[gen] = n.generators_t.p_max_pu[gen].clip(lower=n.generators_t.p_min_pu[gen])

n.optimize(solver_name='highs')

if n.objective is not None:
    print(f'--- Optimized (No Gas Imports) ---')
    print(f'Total System Cost: {n.objective / 1e6:.2f} MEUR')
else:
    print("❌ Optimization failed without gas imports. This suggests the methane demand cannot be met.")

# Results Presentation

## Electricity Generations

#### Specific Profiles

In [ ]:
import matplotlib.pyplot as plt

# --- Extract required generation data from the network ---
hydro_production = n.generators_t.p['SI_Hydro']
hydro_p_nom = n.generators.at['SI_Hydro', 'p_nom']
hydro_max_limit = n.generators_t.p_max_pu['SI_Hydro'] * hydro_p_nom
hydro_min_limit = n.generators_t.p_min_pu['SI_Hydro'] * hydro_p_nom

solar_production = n.generators_t.p['SI_Solar']
wind_production = n.generators_t.p['SI_Wind']
coal_production = n.generators_t.p['SI_Coal']
nuclear_production = n.generators_t.p['SI_Nuclear']

# Create a figure with 5 subplots (5 rows, 1 column), sharing the x-axis
fig, axes = plt.subplots(5, 1, figsize=(15, 18), sharex=True)

# 1. Hydro Profile
axes[0].step(hydro_production.index, hydro_production.values, where='post', color='aqua', label='Hydro Dispatch')
axes[0].step(hydro_max_limit.index, hydro_max_limit.values, where='post', color='lightblue', linestyle='--', label='Max Limit')
axes[0].step(hydro_min_limit.index, hydro_min_limit.values, where='post', color='darkblue', linestyle='--', label='Min Limit')
axes[0].set_title('Hydro Run-of-River Production', fontsize=12)
axes[0].set_ylabel('Power [MW]')
axes[0].legend(loc='upper right')
axes[0].grid(True, alpha=0.3)

# 2. Solar Profile
axes[1].step(solar_production.index, solar_production.values, where='post', color='orange', label='Solar Dispatch')
axes[1].set_title('Solar Production', fontsize=12)
axes[1].set_ylabel('Power [MW]')
axes[1].legend(loc='upper right')
axes[1].grid(True, alpha=0.3)

# 3. Wind Profile
axes[2].step(wind_production.index, wind_production.values, where='post', color='dodgerblue', label='Wind Dispatch')
axes[2].set_title('Wind Production', fontsize=12)
axes[2].set_ylabel('Power [MW]')
axes[2].legend(loc='upper right')
axes[2].grid(True, alpha=0.3)

# 4. Coal Profile
axes[3].step(coal_production.index, coal_production.values, where='post', color='gray', label='Coal Dispatch')
axes[3].set_title('Coal Production', fontsize=12)
axes[3].set_ylabel('Power [MW]')
axes[3].legend(loc='upper right')
axes[3].grid(True, alpha=0.3)

# 5. Nuclear Profile
axes[4].step(nuclear_production.index, nuclear_production.values, where='post', color='purple', label='Nuclear Dispatch')
axes[4].set_title('Nuclear Production', fontsize=12)
axes[4].set_ylabel('Power [MW]')
axes[4].set_xlabel('Date (June 2040)', fontsize=12)
axes[4].legend(loc='upper right')
axes[4].grid(True, alpha=0.3)

# Add a main title for the entire figure
plt.suptitle('Slovenia 2040: Specific Hourly Generation Profiles (June) - Step Format', fontsize=16)
plt.tight_layout()
plt.subplots_adjust(top=0.95)
plt.show()

### Collective Dispaching

In [ ]:
import matplotlib.pyplot as plt

# Extract hourly dispatch for all generators
dispatch = n.generators_t.p.copy()

# Drop the CO2 dummy so it doesn't show up in the graphs
if 'SI_CO2_dummy' in dispatch.columns:
    dispatch = dispatch.drop(columns=['SI_CO2_dummy'])

# --- REORDER FOR STACKING ---
desired_order = ['SI_Nuclear', 'SI_Coal']
other_cols = [c for c in dispatch.columns if c not in desired_order]
final_order = [c for c in desired_order if c in dispatch.columns] + other_cols
dispatch = dispatch[final_order]

# --- STEP FORMAT ADJUSTMENT ---
# Resample to 1-minute intervals and forward fill for the whole month
dispatch_steps = dispatch.resample('1min').ffill()

# Plot the whole month of June
plot_range = slice(n.snapshots[0], n.snapshots[-1] + pd.Timedelta(minutes=59))

# Define color dictionary
gen_colors = {
    'SI_Hydro': 'aqua',
    'SI_Solar': 'orange',
    'SI_Wind': 'dodgerblue',
    'SI_Coal': 'gray',
    'SI_Nuclear': 'purple',
    'SI_gas_imports': 'brown'
}
area_colors = [gen_colors.get(col, 'tab:blue') for col in dispatch.columns]

plt.figure(figsize=(15, 7))
dispatch_steps.loc[plot_range].plot.area(ax=plt.gca(), stacked=True, alpha=0.8, color=area_colors, linewidth=0)

plt.title('Slovenia 2040: Hourly Generator Dispatch (Full Month of June - Step Format)', fontsize=14)
plt.xlabel('Time', fontsize=12)
plt.ylabel('Power Output [MW]', fontsize=12)
plt.legend(title='Generator', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Energy Mix Pie remains the same
energy_mix = (dispatch.sum() * 1e-3).groupby(n.generators.loc[dispatch.columns, 'carrier']).sum()
print("--- June Energy Production by Carrier [GWh] ---")
print(energy_mix.round(2))

carrier_colors = {'hydro': 'aqua', 'solar': 'orange', 'wind': 'dodgerblue', 'coal': 'gray', 'nuclear': 'purple', 'CH4': 'brown'}
pie_colors = [carrier_colors.get(c, 'tab:blue') for c in energy_mix.index]
plt.figure(figsize=(8, 8))
energy_mix.plot.pie(autopct='%1.1f%%', startangle=90, colors=pie_colors)
plt.title('Slovenia 2040: Optimized June Energy Mix')
plt.ylabel('')
plt.show()

## Electricity Price

### Market Price - Day

In [ ]:
import pandas as pd

# Define the specific time slot to analyze
target_time = '2040-06-10 13:00:00'

# 1. Marginal Price (Clearing Price)
price = n.buses_t.marginal_price.loc[target_time, 'SI_elec']
print(f"--- Market Clearing for SI_elec at {target_time} ---")
print(f"Clearing Price: {price:.2f} EUR/MWh\n")

# 2. Supply (Production & Storage Discharging)
print("--- SUPPLY (Generation & Injections) ---")
supply_total = 0

# Generators
for gen in n.generators[n.generators.bus == 'SI_elec'].index:
    p_out = n.generators_t.p.loc[target_time, gen]
    if p_out > 0.01:
        mc = n.generators.at[gen, 'marginal_cost']
        print(f"{gen:20}: {p_out:8.2f} MW  (Marginal Cost: {mc:5.2f} EUR/MWh)")
        supply_total += p_out

# Storage units (positive p means discharging)
for su in n.storage_units[n.storage_units.bus == 'SI_elec'].index:
    p_out = n.storage_units_t.p.loc[target_time, su]
    if p_out > 0.01:
        print(f"{su:20}: {p_out:8.2f} MW  (Storage Discharging)")
        supply_total += p_out

# Links supplying electricity (bus1 == 'SI_elec')
# In PyPSA, p1 is the power *withdrawn* from bus1, so -p1 is injected *into* bus1.
for link in n.links[n.links.bus1 == 'SI_elec'].index:
    p_injected = -n.links_t.p1.loc[target_time, link]
    if p_injected > 0.01:
        print(f"{link:20}: {p_injected:8.2f} MW  (Link Supply)")
        supply_total += p_injected

print(f"\n{'Total Supply':20}: {supply_total:8.2f} MW\n")

# 3. Demand (Consumption)
print("--- DEMAND (Load, Storage Charging & Sector Coupling) ---")
demand_total = 0

# Fixed Loads
for load in n.loads[n.loads.bus == 'SI_elec'].index:
    if load in n.loads_t.p.columns:
        p_in = n.loads_t.p.loc[target_time, load]
    else:
        p_in = n.loads.at[load, 'p_set']
        if isinstance(p_in, pd.Series):  # safety check if load is dynamic but not in p
            p_in = p_in.loc[target_time]
    if p_in > 0.01:
        print(f"{load:20}: {p_in:8.2f} MW  (Fixed Load)")
        demand_total += p_in

# Storage units (negative p means charging)
for su in n.storage_units[n.storage_units.bus == 'SI_elec'].index:
    p_out = n.storage_units_t.p.loc[target_time, su]
    if p_out < -0.01:
        print(f"{su:20}: {-p_out:8.2f} MW  (Storage Charging)")
        demand_total += -p_out

# Links consuming electricity (bus0 == 'SI_elec')
for link in n.links[n.links.bus0 == 'SI_elec'].index:
    p_consumed = n.links_t.p0.loc[target_time, link]
    if p_consumed > 0.01:
        print(f"{link:20}: {p_consumed:8.2f} MW  (Link Consumption)")
        demand_total += p_consumed

print(f"\n{'Total Demand':20}: {demand_total:8.2f} MW")
print(f"{'Balance (Supply-Dem)':20}: {supply_total - demand_total:8.2f} MW")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

target_time = '2040-06-10 13:00:00'
price = n.buses_t.marginal_price.loc[target_time, 'SI_elec']

# --- 1. Gather Data for Bar Chart ---
supply_data = {}
demand_data = {}

# Supply
for gen in n.generators[n.generators.bus == 'SI_elec'].index:
    p_out = n.generators_t.p.loc[target_time, gen]
    if p_out > 0.01: supply_data[gen] = p_out
for su in n.storage_units[n.storage_units.bus == 'SI_elec'].index:
    p_out = n.storage_units_t.p.loc[target_time, su]
    if p_out > 0.01: supply_data[f"{su} (Discharge)"] = p_out
for link in n.links[n.links.bus1 == 'SI_elec'].index:
    p_injected = -n.links_t.p1.loc[target_time, link]
    if p_injected > 0.01: supply_data[f"{link} (Supply)"] = p_injected

# Demand
demand_total = 0
for load in n.loads[n.loads.bus == 'SI_elec'].index:
    if load in n.loads_t.p.columns: p_in = n.loads_t.p.loc[target_time, load]
    else:
        p_in = n.loads.at[load, 'p_set']
        if isinstance(p_in, pd.Series): p_in = p_in.loc[target_time]
    if p_in > 0.01:
        demand_data[load] = p_in
        demand_total += p_in
for su in n.storage_units[n.storage_units.bus == 'SI_elec'].index:
    p_out = n.storage_units_t.p.loc[target_time, su]
    if p_out < -0.01:
        demand_data[f"{su} (Charge)"] = -p_out
        demand_total += -p_out
for link in n.links[n.links.bus0 == 'SI_elec'].index:
    p_consumed = n.links_t.p0.loc[target_time, link]
    if p_consumed > 0.01:
        demand_data[f"{link} (Consumption)"] = p_consumed
        demand_total += p_consumed

# --- 2. Gather Data for Merit Order (Nuclear first) ---
offers_list = []
for gen in n.generators[n.generators.bus == 'SI_elec'].index:
    if gen in n.generators_t.p_max_pu.columns:
        cap = n.generators_t.p_max_pu.loc[target_time, gen] * n.generators.at[gen, 'p_nom']
    else:
        cap = n.generators.at[gen, 'p_max_pu'] * n.generators.at[gen, 'p_nom']
    mc = n.generators.at[gen, 'marginal_cost']
    if cap > 0.1:
        # Force Nuclear to have a slightly lower cost for sorting, or specific flag
        sort_mc = -1.0 if gen == 'SI_Nuclear' else mc
        offers_list.append({'Generator': gen, 'Capacity_MW': cap, 'Marginal_Cost': mc, 'Sort_MC': sort_mc})

df_offers = pd.DataFrame(offers_list).sort_values(by='Sort_MC')
df_offers['Cumulative_Capacity_MW'] = df_offers['Capacity_MW'].cumsum()

# --- Color Mapping ---
item_colors = {
    'SI_Hydro': 'aqua',
    'SI_Solar': 'orange',
    'SI_Wind': 'dodgerblue',
    'SI_Coal': 'gray',
    'SI_Nuclear': 'purple',
    'SI_gas_imports': 'brown',
    'SI_battery (Discharge)': 'lightgreen',
    'SI_battery (Charge)': 'lightgreen',
    'SI_elec_load': 'brown',
    'SI_electrolyzer (Consumption)': 'teal'
}

# --- Plotting ---
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(15, 14))

# -- Subplot 1: Stacked Bar Chart --
bottom_supply = 0
for key, value in supply_data.items():
    c = item_colors.get(key, 'tab:blue')
    ax1.bar('Supply', value, bottom=bottom_supply, label=key, color=c, edgecolor='white', linewidth=1)
    ax1.text(0, bottom_supply + value/2, f"{value:.0f} MW", ha='center', va='center', color='black', fontsize=9, fontweight='bold', bbox=dict(facecolor='white', alpha=0.5, edgecolor='none', pad=1))
    bottom_supply += value

bottom_demand = 0
for key, value in demand_data.items():
    c = item_colors.get(key, 'tab:blue')
    ax1.bar('Demand', value, bottom=bottom_demand, label=key, color=c, edgecolor='white', linewidth=1)
    ax1.text(1, bottom_demand + value/2, f"{value:.0f} MW", ha='center', va='center', color='white', fontsize=9, fontweight='bold', bbox=dict(facecolor='black', alpha=0.5, edgecolor='none', pad=1))
    bottom_demand += value

ax1.set_title(f'Supply & Demand Balance', fontsize=14)
ax1.set_ylabel('Power [MW]', fontsize=12)
ax1.legend(title='Components', bbox_to_anchor=(1.05, 1), loc='upper left')
ax1.grid(axis='y', linestyle='--', alpha=0.5)

# -- Subplot 2: Merit Order Curve --
x = [0]
y = [df_offers['Marginal_Cost'].iloc[0]] if not df_offers.empty else [0]
for idx, row in df_offers.iterrows():
    x.append(row['Cumulative_Capacity_MW'])
    y.append(row['Marginal_Cost'])

ax2.step(x, y, where='pre', color='blue', linewidth=2, label='Supply Curve (Merit Order)')

prev_x = 0
for idx, row in df_offers.iterrows():
    c = item_colors.get(row['Generator'], 'tab:blue')
    ax2.fill_between([prev_x, row['Cumulative_Capacity_MW']], 0, row['Marginal_Cost'],
                     step='pre', alpha=0.5, color=c)
    mid_x = prev_x + (row['Capacity_MW'] / 2)
    ax2.text(mid_x, row['Marginal_Cost'] + 2, row['Generator'], rotation=90, ha='center', va='bottom', fontsize=10)
    prev_x = row['Cumulative_Capacity_MW']

ax2.axvline(x=demand_total, color='red', linestyle='--', linewidth=2, label=f'Total Demand ({demand_total:.0f} MW)')
ax2.axhline(y=price, color='green', linestyle=':', linewidth=2, label=f'Clearing Price ({price:.2f} EUR/MWh)')

ax2.set_title(f'Merit Order Curve (Nuclear Must-Run Forced to Left)', fontsize=14)
ax2.set_xlabel('Capacity / Demand [MW]', fontsize=12)
ax2.set_ylabel('Marginal Cost / Price [EUR/MWh]', fontsize=12)
ax2.set_xlim(0, max(x) + 500)
ax2.set_ylim(0, max(max(y), price) + 20)
ax2.grid(True, alpha=0.3)
ax2.legend(loc='upper left')

plt.suptitle(f'Slovenia 2040: Market Clearing Analysis at {target_time}\nClearing Price: {price:.2f} EUR/MWh', fontsize=16)
plt.tight_layout()
plt.subplots_adjust(top=0.92)
plt.show()

### Market price - Night

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

target_time = '2040-06-13 01:00:00'
price = n.buses_t.marginal_price.loc[target_time, 'SI_elec']

# --- 1. Gather Data for Bar Chart ---
supply_data = {}
demand_data = {}

# Supply
for gen in n.generators[n.generators.bus == 'SI_elec'].index:
    p_out = n.generators_t.p.loc[target_time, gen]
    if p_out > 0.01: supply_data[gen] = p_out
for su in n.storage_units[n.storage_units.bus == 'SI_elec'].index:
    p_out = n.storage_units_t.p.loc[target_time, su]
    if p_out > 0.01: supply_data[f"{su} (Discharge)"] = p_out
for link in n.links[n.links.bus1 == 'SI_elec'].index:
    p_injected = -n.links_t.p1.loc[target_time, link]
    if p_injected > 0.01: supply_data[f"{link} (Supply)"] = p_injected

# Demand
demand_total = 0
for load in n.loads[n.loads.bus == 'SI_elec'].index:
    if load in n.loads_t.p.columns: p_in = n.loads_t.p.loc[target_time, load]
    else:
        p_in = n.loads.at[load, 'p_set']
        if isinstance(p_in, pd.Series): p_in = p_in.loc[target_time]
    if p_in > 0.01:
        demand_data[load] = p_in
        demand_total += p_in
for su in n.storage_units[n.storage_units.bus == 'SI_elec'].index:
    p_out = n.storage_units_t.p.loc[target_time, su]
    if p_out < -0.01:
        demand_data[f"{su} (Charge)"] = -p_out
        demand_total += -p_out
for link in n.links[n.links.bus0 == 'SI_elec'].index:
    p_consumed = n.links_t.p0.loc[target_time, link]
    if p_consumed > 0.01:
        demand_data[f"{link} (Consumption)"] = p_consumed
        demand_total += p_consumed

# --- 2. Gather Data for Merit Order ---
offers = []
for gen in n.generators[n.generators.bus == 'SI_elec'].index:
    if gen in n.generators_t.p_max_pu.columns:
        cap = n.generators_t.p_max_pu.loc[target_time, gen] * n.generators.at[gen, 'p_nom']
    else:
        cap = n.generators.at[gen, 'p_max_pu'] * n.generators.at[gen, 'p_nom']
    mc = n.generators.at[gen, 'marginal_cost']
    if cap > 0.1:
        offers.append({'Generator': gen, 'Capacity_MW': cap, 'Marginal_Cost': mc})

df_offers = pd.DataFrame(offers).sort_values(by='Marginal_Cost')
df_offers['Cumulative_Capacity_MW'] = df_offers['Capacity_MW'].cumsum()

# --- Color Mapping ---
item_colors = {
    'SI_Hydro_ror': 'aqua',
    'SI_Solar': 'orange',
    'SI_Wind': 'dodgerblue',
    'SI_Coal': 'gray',
    'SI_Nuclear': 'purple',
    'SI_gas_imports': 'brown',
    'SI_battery (Discharge)': 'lightgreen',
    'SI_battery (Charge)': 'lightgreen',
    'SI_elec_load': 'brown',
    'SI_electrolyzer (Consumption)': 'teal'
}

# --- Plotting ---
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(15, 14))

# -- Subplot 1: Stacked Bar Chart --
bottom_supply = 0
for key, value in supply_data.items():
    c = item_colors.get(key, 'tab:blue')
    ax1.bar('Supply', value, bottom=bottom_supply, label=key, color=c, edgecolor='white', linewidth=1)
    ax1.text(0, bottom_supply + value/2, f"{value:.0f} MW", ha='center', va='center', color='black', fontsize=9, fontweight='bold', bbox=dict(facecolor='white', alpha=0.5, edgecolor='none', pad=1))
    bottom_supply += value

bottom_demand = 0
for key, value in demand_data.items():
    c = item_colors.get(key, 'tab:blue')
    ax1.bar('Demand', value, bottom=bottom_demand, label=key, color=c, edgecolor='white', linewidth=1)
    ax1.text(1, bottom_demand + value/2, f"{value:.0f} MW", ha='center', va='center', color='white', fontsize=9, fontweight='bold', bbox=dict(facecolor='black', alpha=0.5, edgecolor='none', pad=1))
    bottom_demand += value

ax1.set_title(f'Supply & Demand Balance', fontsize=14)
ax1.set_ylabel('Power [MW]', fontsize=12)
ax1.legend(title='Components', bbox_to_anchor=(1.05, 1), loc='upper left')
ax1.grid(axis='y', linestyle='--', alpha=0.5)

# -- Subplot 2: Merit Order Curve --
x = [0]
y = [df_offers['Marginal_Cost'].iloc[0]] if not df_offers.empty else [0]
for idx, row in df_offers.iterrows():
    x.append(row['Cumulative_Capacity_MW'])
    y.append(row['Marginal_Cost'])

ax2.step(x, y, where='pre', color='blue', linewidth=2, label='Supply Curve (Merit Order)')

prev_x = 0
for idx, row in df_offers.iterrows():
    c = item_colors.get(row['Generator'], 'tab:blue')
    ax2.fill_between([prev_x, row['Cumulative_Capacity_MW']], 0, row['Marginal_Cost'],
                     step='pre', alpha=0.5, color=c)
    mid_x = prev_x + (row['Capacity_MW'] / 2)
    ax2.text(mid_x, row['Marginal_Cost'] + 2, row['Generator'], rotation=90, ha='center', va='bottom', fontsize=10)
    prev_x = row['Cumulative_Capacity_MW']

ax2.axvline(x=demand_total, color='red', linestyle='--', linewidth=2, label=f'Total Demand ({demand_total:.0f} MW)')
ax2.axhline(y=price, color='green', linestyle=':', linewidth=2, label=f'Clearing Price ({price:.2f} EUR/MWh)')

ax2.set_title(f'Merit Order Curve', fontsize=14)
ax2.set_xlabel('Capacity / Demand [MW]', fontsize=12)
ax2.set_ylabel('Marginal Cost / Price [EUR/MWh]', fontsize=12)
ax2.set_xlim(0, max(x) + 500)
ax2.set_ylim(0, max(max(y), price) + 20)
ax2.grid(True, alpha=0.3)
ax2.legend(loc='upper left')

plt.suptitle(f'Slovenia 2040: Market Clearing Analysis at {target_time}\nClearing Price: {price:.2f} EUR/MWh', fontsize=16)
plt.tight_layout()
plt.subplots_adjust(top=0.92)
plt.show()


### Electricity Price in Time

In [ ]:
import matplotlib.pyplot as plt

# Extract prices for the full month of June (30 days)
start_date = '2040-06-01'
end_date = '2040-06-30 23:00:00'
prices_june = n.buses_t.marginal_price.loc[start_date:end_date, 'SI_elec']

# Plot the prices
plt.figure(figsize=(15, 6))
plt.plot(prices_june.index, prices_june.values, color='k', linewidth=1.5)

plt.title('Slovenia 2040: Electricity Marginal Prices (Full Month of June)', fontsize=14)
plt.ylabel('Price [EUR/MWh]')
plt.xlabel('Date')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Print statistics
print(f"--- Statistics for June (30 Days) ---")
print(f"Average Price: {prices_june.mean():.2f} EUR/MWh")
print(f"Max Price:     {prices_june.max():.2f} EUR/MWh")
print(f"Min Price:     {prices_june.min():.2f} EUR/MWh")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

target_time = '2040-06-19 17:00:00'
price = n.buses_t.marginal_price.loc[target_time, 'SI_elec']

# --- 1. Gather Data for Bar Chart ---
supply_data = {}
demand_data = {}

# Supply
for gen in n.generators[n.generators.bus == 'SI_elec'].index:
    p_out = n.generators_t.p.loc[target_time, gen]
    if p_out > 0.01: supply_data[gen] = p_out
for su in n.storage_units[n.storage_units.bus == 'SI_elec'].index:
    p_out = n.storage_units_t.p.loc[target_time, su]
    if p_out > 0.01: supply_data[f"{su} (Discharge)"] = p_out
for link in n.links[n.links.bus1 == 'SI_elec'].index:
    p_injected = -n.links_t.p1.loc[target_time, link]
    if p_injected > 0.01: supply_data[f"{link} (Supply)"] = p_injected

# Demand
demand_total = 0
for load in n.loads[n.loads.bus == 'SI_elec'].index:
    if load in n.loads_t.p.columns: p_in = n.loads_t.p.loc[target_time, load]
    else:
        p_in = n.loads.at[load, 'p_set']
        if isinstance(p_in, pd.Series): p_in = p_in.loc[target_time]
    if p_in > 0.01:
        demand_data[load] = p_in
        demand_total += p_in
for su in n.storage_units[n.storage_units.bus == 'SI_elec'].index:
    p_out = n.storage_units_t.p.loc[target_time, su]
    if p_out < -0.01:
        demand_data[f"{su} (Charge)"] = -p_out
        demand_total += -p_out
for link in n.links[n.links.bus0 == 'SI_elec'].index:
    p_consumed = n.links_t.p0.loc[target_time, link]
    if p_consumed > 0.01:
        demand_data[f"{link} (Consumption)"] = p_consumed
        demand_total += p_consumed

# --- 2. Gather Data for Merit Order ---
offers = []
for gen in n.generators[n.generators.bus == 'SI_elec'].index:
    if gen in n.generators_t.p_max_pu.columns:
        cap = n.generators_t.p_max_pu.loc[target_time, gen] * n.generators.at[gen, 'p_nom']
    else:
        cap = n.generators.at[gen, 'p_max_pu'] * n.generators.at[gen, 'p_nom']
    mc = n.generators.at[gen, 'marginal_cost']
    if cap > 0.1:
        offers.append({'Generator': gen, 'Capacity_MW': cap, 'Marginal_Cost': mc})

df_offers = pd.DataFrame(offers).sort_values(by='Marginal_Cost')
df_offers['Cumulative_Capacity_MW'] = df_offers['Capacity_MW'].cumsum()

# --- Color Mapping ---
item_colors = {
    'SI_Hydro_ror': 'aqua',
    'SI_Solar': 'orange',
    'SI_Wind': 'dodgerblue',
    'SI_Coal': 'gray',
    'SI_Nuclear': 'purple',
    'SI_gas_imports': 'brown',
    'SI_battery (Discharge)': 'lightgreen',
    'SI_battery (Charge)': 'lightgreen',
    'SI_elec_load': 'brown',
    'SI_electrolyzer (Consumption)': 'teal'
}

# --- Plotting ---
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(15, 14))

# -- Subplot 1: Stacked Bar Chart --
bottom_supply = 0
for key, value in supply_data.items():
    c = item_colors.get(key, 'tab:blue')
    ax1.bar('Supply', value, bottom=bottom_supply, label=key, color=c, edgecolor='white', linewidth=1)
    ax1.text(0, bottom_supply + value/2, f"{value:.0f} MW", ha='center', va='center', color='black', fontsize=9, fontweight='bold', bbox=dict(facecolor='white', alpha=0.5, edgecolor='none', pad=1))
    bottom_supply += value

bottom_demand = 0
for key, value in demand_data.items():
    c = item_colors.get(key, 'tab:blue')
    ax1.bar('Demand', value, bottom=bottom_demand, label=key, color=c, edgecolor='white', linewidth=1)
    ax1.text(1, bottom_demand + value/2, f"{value:.0f} MW", ha='center', va='center', color='white', fontsize=9, fontweight='bold', bbox=dict(facecolor='black', alpha=0.5, edgecolor='none', pad=1))
    bottom_demand += value

ax1.set_title(f'Supply & Demand Balance', fontsize=14)
ax1.set_ylabel('Power [MW]', fontsize=12)
ax1.legend(title='Components', bbox_to_anchor=(1.05, 1), loc='upper left')
ax1.grid(axis='y', linestyle='--', alpha=0.5)

# -- Subplot 2: Merit Order Curve --
x = [0]
y = [df_offers['Marginal_Cost'].iloc[0]] if not df_offers.empty else [0]
for idx, row in df_offers.iterrows():
    x.append(row['Cumulative_Capacity_MW'])
    y.append(row['Marginal_Cost'])

ax2.step(x, y, where='pre', color='blue', linewidth=2, label='Supply Curve (Merit Order)')

prev_x = 0
for idx, row in df_offers.iterrows():
    c = item_colors.get(row['Generator'], 'tab:blue')
    ax2.fill_between([prev_x, row['Cumulative_Capacity_MW']], 0, row['Marginal_Cost'],
                     step='pre', alpha=0.5, color=c)
    mid_x = prev_x + (row['Capacity_MW'] / 2)
    ax2.text(mid_x, row['Marginal_Cost'] + 2, row['Generator'], rotation=90, ha='center', va='bottom', fontsize=10)
    prev_x = row['Cumulative_Capacity_MW']

ax2.axvline(x=demand_total, color='red', linestyle='--', linewidth=2, label=f'Total Demand ({demand_total:.0f} MW)')
ax2.axhline(y=price, color='green', linestyle=':', linewidth=2, label=f'Clearing Price ({price:.2f} EUR/MWh)')

ax2.set_title(f'Merit Order Curve', fontsize=14)
ax2.set_xlabel('Capacity / Demand [MW]', fontsize=12)
ax2.set_ylabel('Marginal Cost / Price [EUR/MWh]', fontsize=12)
ax2.set_xlim(0, max(x) + 500)
ax2.set_ylim(0, max(max(y), price) + 20)
ax2.grid(True, alpha=0.3)
ax2.legend(loc='upper left')

plt.suptitle(f'Slovenia 2040: Market Clearing Analysis at {target_time}\nClearing Price: {price:.2f} EUR/MWh', fontsize=16)
plt.tight_layout()
plt.subplots_adjust(top=0.92)
plt.show()


## Total electricity System

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# --- 1. Data Preparation ---
# Electricity Data
gens_to_sum = ['SI_Nuclear', 'SI_Coal', 'SI_Hydro', 'SI_Solar', 'SI_Wind']
total_generation = n.generators_t.p[gens_to_sum].sum(axis=1)
demand = n.loads_t.p['SI_elec_load']

# Battery Data
battery_dispatch = n.storage_units_t.p['SI_battery']
battery_soc = n.storage_units_t.state_of_charge['SI_battery']

# Hydrogen Interface Data (p0 is power withdrawal, p1 is injection)
elec_to_h2 = n.links_t.p0['SI_electrolyzer']
h2_to_elec = -n.links_t.p1['SI_h2_to_power']

# --- 2. Step Format Resampling ---
total_gen_steps = total_generation.resample('1min').ffill()
demand_steps = demand.resample('1min').ffill()
batt_p_steps = battery_dispatch.resample('1min').ffill()
batt_soc_steps = battery_soc.resample('1min').ffill()
elec_to_h2_steps = elec_to_h2.resample('1min').ffill()
h2_to_elec_steps = h2_to_elec.resample('1min').ffill()

# --- 3. Plotting ---
fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(15, 18), sharex=True)

# Subplot 1: Electricity Balance
ax1.plot(total_gen_steps.index, total_gen_steps.values, label='Total Generation (5 Sources)', color='darkblue', linewidth=2, alpha=0.8)
ax1.plot(demand_steps.index, demand_steps.values, label='Electricity Demand', color='brown', linestyle='--', linewidth=2)
ax1.set_title('Slovenia 2040: Total Generation vs. Electricity Demand (June)', fontsize=14)
ax1.set_ylabel('Power [MW]')
ax1.grid(True, alpha=0.3)
ax1.legend(loc='upper right')

# Subplot 2: Battery Dashboard
ax2.fill_between(batt_soc_steps.index, 0, batt_soc_steps.values, color='lightgreen', alpha=0.4, label='Battery SoC [MWh]')
ax2.set_ylabel('State of Charge [MWh]', color='darkgreen')
ax2.tick_params(axis='y', labelcolor='darkgreen')
ax2_p = ax2.twinx()
ax2_p.step(batt_p_steps.index, batt_p_steps.values, where='post', color='darkgreen', linewidth=1.0, label='Battery Power (Charge - / Discharge +)')
ax2_p.set_ylabel('Battery Power Flow [MW]', color='blue')
ax2_p.tick_params(axis='y', labelcolor='blue')
ax2_p.axhline(0, color='black', linewidth=0.8, alpha=0.5)
ax2.set_title('Slovenia 2040: Battery State of Charge and Power Flow', fontsize=14)
ax2.grid(True, alpha=0.3)
lines_a, labels_a = ax2.get_legend_handles_labels()
lines_b, labels_b = ax2_p.get_legend_handles_labels()
ax2.legend(lines_a + lines_b, labels_a + labels_b, loc='upper right')

# Subplot 3: Hydrogen Market Interface
ax3.step(elec_to_h2_steps.index, elec_to_h2_steps.values, where='post', color='teal', label='Electrolyzer (Elec to H2)', linewidth=2)
ax3.step(h2_to_elec_steps.index, h2_to_elec_steps.values, where='post', color='darkblue', label='Generation (H2 to Elec)', linewidth=2)
ax3.set_ylabel('Power Flow [MW]')
ax3.set_title('Slovenia 2040: Hydrogen & Electricity Market Interfaces', fontsize=14)
ax3.set_xlabel('Date')
ax3.grid(True, alpha=0.3)
ax3.legend(loc='upper right')

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# 1. Extract Interface Data
# Electricity to H2 (Electrolyzer consumption)
elec_to_h2 = n.links_t.p0['SI_electrolyzer']
# H2 to Electricity (Fuel cell / Turbine generation)
h2_to_elec = -n.links_t.p1['SI_h2_to_power']

# 2. Resample for step format
elec_to_h2_steps = elec_to_h2.resample('1min').ffill()
h2_to_elec_steps = h2_to_elec.resample('1min').ffill()

# 3. Plotting
plt.figure(figsize=(15, 7))

# Plotting interfaces
plt.step(elec_to_h2_steps.index, elec_to_h2_steps.values, where='post', color='teal', label='Electricity to H2 (Electrolyzer Consumption)', linewidth=2)
plt.step(h2_to_elec_steps.index, h2_to_elec_steps.values, where='post', color='darkcyan', label='H2 to Electricity (Generation)', linewidth=2)

# Add total H2 demand reference if available
h2_demand_constant = n.loads.at['SI_h2_load', 'p_set'] if 'SI_h2_load' in n.loads.index else 0
plt.axhline(h2_demand_constant, color='black', linestyle=':', alpha=0.6, label='Constant H2 Demand Ref')

plt.title('Slovenia 2040: Hydrogen & Electricity Market Interfaces (June)', fontsize=14)
plt.ylabel('Power Flow [MW]')
plt.xlabel('Date')
plt.legend(loc='upper right')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"--- June Hydrogen Interface Totals ---")
print(f"Electricity Consumed for H2: {elec_to_h2.sum()/1e3:.2f} GWh_elec")
print(f"Electricity Generated from H2: {h2_to_elec.sum()/1e3:.2f} GWh_elec")

### Hydrogen and Methane Consumption

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# 1. Hydrogen Balance (Annual)
# Identify available links for H2 production
h2_link_list = [l for l in ['SI_electrolyzer', 'SI_SMR', 'SI_SMR_with_CCS'] if l in n.links.index]
h2_prod = n.links_t.p1[h2_link_list].sum(axis=1) if h2_link_list else pd.Series(0, index=n.snapshots)

# Load handling
h2_load = n.loads_t.p_set['SI_h2_load'] if 'SI_h2_load' in n.loads_t.p_set.columns else pd.Series(0, index=n.snapshots)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(15, 10))

ax1.plot(n.snapshots, h2_prod, label='H2 Production [MW]', color='teal', alpha=0.7)
ax1.plot(n.snapshots, h2_load, label='H2 Demand [MW]', color='black', linestyle='--')
ax1.set_title('Slovenia 2040: Annual Hydrogen Market Balance', fontsize=14)
ax1.legend()
ax1.grid(True, alpha=0.3)

# Only plot SoC if storage exists
if 'SI_h2_storage' in n.stores.index:
    h2_soc = n.stores_t.e['SI_h2_storage']
    ax1_twin = ax1.twinx()
    ax1_twin.fill_between(n.snapshots, 0, h2_soc, color='orange', alpha=0.2, label='Storage SoC [MWh]')
    ax1_twin.set_ylabel('Storage Energy [MWh]')
    ax1_twin.legend(loc='lower right')

# 2. Methane Balance (Annual)
gas_link_list = [l for l in ['SI_biomethane_injection', 'SI_methanation'] if l in n.links.index]
gas_supply = n.links_t.p1[gas_link_list].sum(axis=1) if gas_link_list else pd.Series(0, index=n.snapshots)
if 'SI_gas_imports' in n.generators.index:
    gas_supply += n.generators_t.p['SI_gas_imports']

gas_load = n.loads_t.p_set['SI_methane_load'] if 'SI_methane_load' in n.loads_t.p_set.columns else pd.Series(0, index=n.snapshots)

ax2.plot(n.snapshots, gas_supply, label='CH4 Supply/Imports [MW]', color='brown', alpha=0.7)
ax2.plot(n.snapshots, gas_load, label='CH4 Demand [MW]', color='black', linestyle='--')
ax2.set_title('Slovenia 2040: Annual Methane Market and Seasonal Storage', fontsize=14)
ax2.legend()
ax2.grid(True, alpha=0.3)

if 'SI_methane_storage' in n.stores.index:
    gas_soc = n.stores_t.e['SI_methane_storage']
    ax2_twin = ax2.twinx()
    ax2_twin.fill_between(n.snapshots, 0, gas_soc, color='green', alpha=0.2, label='Storage SoC [MWh]')
    ax2_twin.set_ylabel('Storage Energy [MWh]')
    ax2_twin.legend(loc='lower right')

plt.tight_layout()
plt.show()